# API Functions → 7-Class Malware Classifier

Trains a Random Forest on API function presence features from the Windows Malware Detection dataset:
- **Features**: ~21,900 binary columns (1 if the API function is present, 0 otherwise)
- **Labels**: `Type` 0–6 (0 = benign, 1–6 = malware subtypes)
- **Model**: `RandomForestClassifier` (53 trees), mirroring [`ZEIT8025_ML_Malware.ipynb`](ZEIT8025_ML_Malware.ipynb)

Run all cells in order. Data is downloaded from Azure Blob Storage.

> **Note**: This dataset is wide (~22k features). Training may take several minutes and requires sufficient Colab RAM (~2 GB for the feature matrix).

## Step 1 — Imports

In [ ]:
import os
import requests
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)

print(f"pandas {pd.__version__}")

## Step 2 — Download CSV from URL

Source: Azure Blob Storage (`API_Functions.csv`).

> If download fails, upload `API_Functions.csv` to `/content/API_Functions.csv` manually.

In [ ]:
CSV_URL   = (
    "https://pocstorage1705.blob.core.windows.net/storage/input/API_Functions.csv"
    "?sp=r&st=2026-06-09T12:50:32Z&se=2027-06-09T21:05:32Z"
    "&spr=https&sv=2026-02-06&sr=b&sig=2EZHjbs99uuT996FPN4xAdWN32oShGzmPymQrkTjIo4%3D"
)
LOCAL_CSV = "/content/API_Functions.csv"

if os.path.exists(LOCAL_CSV):
    print(f"Already cached: {LOCAL_CSV}")
else:
    print("Downloading API_Functions.csv …")
    with requests.get(CSV_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(LOCAL_CSV, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {LOCAL_CSV}  ({os.path.getsize(LOCAL_CSV) / 1e6:.1f} MB)")

## Step 3 — Load and Inspect

In [ ]:
df = pd.read_csv(LOCAL_CSV, low_memory=False)
print(df.info())
print(f"\nLabel distribution (Type):")
print(df["Type"].value_counts().sort_index().rename("sample_count").to_string())

## Step 4 — Prepare Features and Labels

Drop `SHA256` and `Type`. All remaining columns are binary API-presence indicators (0/1). Features are stored as `int8` to reduce memory use.

In [ ]:
labels = df["Type"].values
feature_cols = [c for c in df.columns if c not in ("SHA256", "Type")]
features = df[feature_cols].astype("int8").values

print(f"Features shape : {features.shape}")
print(f"Labels shape   : {labels.shape}")
print(f"Unique classes : {sorted(set(labels))}")
print(f"Feature memory : {features.nbytes / 1e6:.1f} MB")

## Step 5 — Train / Test Split and Model Training

80/20 stratified split; `RandomForestClassifier` with 53 estimators (same as the ClaMP reference notebook).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

model = RandomForestClassifier(n_estimators=53, random_state=42)
model.fit(X_train, y_train)
predicted = model.predict(X_test)

print(f"Train samples : {len(y_train):,}")
print(f"Test samples  : {len(y_test):,}")

## Step 6 — Evaluate

Multi-class metrics with `average="weighted"`.

In [ ]:
cm = confusion_matrix(y_test, predicted)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()

accuracy  = accuracy_score(y_test, predicted)
precision = precision_score(y_test, predicted, average="weighted")
recall    = recall_score(y_test, predicted, average="weighted")
f1        = f1_score(y_test, predicted, average="weighted")

print("Accuracy:  ", accuracy)
print("Precision: ", precision)
print("Recall:    ", recall)
print("F1 Score:  ", f1)

## (Optional) Preview Predictions

In [ ]:
preview = pd.DataFrame({"actual": y_test, "predicted": predicted})
preview.head(10)